# 02 — Speaker-independent split, 10-fold assignments, and MFCC preprocessing

Run after completing the dataset manifest in Notebook 01. This notebook reserves a speaker-independent final test set and creates 10 grouped folds using only the remaining training speakers.

**Important:** all files belonging to one speaker stay in one partition. Original recordings and their augmented derivatives are also kept together.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
N_SPLITS = 10
TEST_SIZE = 0.20

OUT = Path("anu_revision_output")
OUT.mkdir(exist_ok=True)

manifest_path = OUT / "dataset_manifest_TO_FILL.csv"
df = pd.read_csv(manifest_path).fillna("")

required = ["path", "label", "speaker_id", "original_file", "is_augmented"]
for col in required:
    assert col in df.columns, f"Missing manifest column: {col}"
    assert (df[col].astype(str).str.strip() != "").all(), f"Complete column: {col}"

assert df["path"].map(lambda p: Path(p).exists()).all(), "Some WAV paths do not exist."
assert (df.groupby("original_file")["speaker_id"].nunique() == 1).all(), (
    "An original recording and its derivatives have inconsistent speaker IDs."
)

df["is_augmented"] = df["is_augmented"].astype(str).str.lower().isin(["true", "1", "yes"])
print("Audio files:", len(df))
print("Classes:", df["label"].nunique())
print("Speaker IDs:", df["speaker_id"].nunique())
display(df.groupby("label").size().to_frame("audio_count"))

Audio files: 4440
Classes: 14
Speaker IDs: 53


,audio_count
label,
anu,318
close browser,324
close calculator,324
help,318
mute,288
next,318
open browser,318
open calculator,324
pause,318


In [2]:
# Reserve the final test speakers once. These speakers are not used in cross-validation.
outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(outer_split.split(df, groups=df["speaker_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert set(train_df["speaker_id"]).isdisjoint(set(test_df["speaker_id"]))
assert set(train_df["original_file"]).isdisjoint(set(test_df["original_file"]))

print("Training speaker IDs:", train_df["speaker_id"].nunique())
print("Final-test speaker IDs:", test_df["speaker_id"].nunique())
print("Training files:", len(train_df))
print("Final-test files:", len(test_df))
display(test_df.groupby("label").size().to_frame("test_count"))

Training speaker IDs: 42
Final-test speaker IDs: 11
Training files: 3522
Final-test files: 918


,test_count
label,
anu,66
close browser,66
close calculator,66
help,66
mute,60
next,66
open browser,66
open calculator,66
pause,66


In [3]:
# Extract MFCC features. Normalization statistics are fitted only on the outer training set.
SR = 22050
N_MFCC = 20
FRAMES = 40

def extract_mfcc(path):
    signal, _ = librosa.load(path, sr=SR, mono=True)
    features = librosa.feature.mfcc(y=signal, sr=SR, n_mfcc=N_MFCC)

    if features.shape[1] < FRAMES:
        features = np.pad(features, ((0, 0), (0, FRAMES - features.shape[1])))
    else:
        features = features[:, :FRAMES]

    return features.astype("float32")

X_train_raw = np.stack([extract_mfcc(path) for path in train_df["path"]])
X_test_raw = np.stack([extract_mfcc(path) for path in test_df["path"]])

mean = float(X_train_raw.mean())
std = float(X_train_raw.std() + 1e-6)

X_train = ((X_train_raw - mean) / std)[..., None]
X_test = ((X_test_raw - mean) / std)[..., None]

encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df["label"])
y_test = encoder.transform(test_df["label"])
classes = encoder.classes_.astype(str)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Classes:", list(classes))
print("Training normalization mean/std:", mean, std)

X_train: (3522, 20, 40, 1)
X_test: (918, 20, 40, 1)
Classes: [np.str_('anu'), np.str_('close browser'), np.str_('close calculator'), np.str_('help'), np.str_('mute'), np.str_('next'), np.str_('open browser'), np.str_('open calculator'), np.str_('pause'), np.str_('play'), np.str_('previous'), np.str_('screenshot'), np.str_('sleep'), np.str_('unmute')]
Training normalization mean/std: -10.976839065551758 78.21414184570312


In [4]:
# Create 10 stratified, speaker-grouped folds on the outer training data only.
# Each speaker appears in exactly one validation fold.
cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_assignments = np.full(len(train_df), -1, dtype=np.int32)
fold_summaries = []

for fold_number, (_, val_indices) in enumerate(
    cv.split(X_train, y_train, groups=train_df["speaker_id"]), start=1
):
    fold_assignments[val_indices] = fold_number

assert (fold_assignments > 0).all(), "Some training examples were not assigned to a fold."

for fold_number in range(1, N_SPLITS + 1):
    val_mask = fold_assignments == fold_number
    fit_mask = ~val_mask

    fit_speakers = set(train_df.loc[fit_mask, "speaker_id"])
    val_speakers = set(train_df.loc[val_mask, "speaker_id"])

    assert fit_speakers.isdisjoint(val_speakers)

    fold_summaries.append({
        "fold": fold_number,
        "train_files": int(fit_mask.sum()),
        "validation_files": int(val_mask.sum()),
        "train_speakers": len(fit_speakers),
        "validation_speakers": len(val_speakers)
    })

fold_summary = pd.DataFrame(fold_summaries)
display(fold_summary)
print("All 10 folds assigned:", sorted(np.unique(fold_assignments).tolist()))

,fold,train_files,validation_files,train_speakers,validation_speakers
0,1,3174,348,38,4
1,2,3108,414,37,5
2,3,3192,330,38,4
3,4,3192,330,38,4
4,5,3180,342,38,4
5,6,3192,330,38,4
6,7,3180,342,38,4
7,8,3108,414,37,5
8,9,3186,336,38,4
9,10,3186,336,38,4


All 10 folds assigned: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [5]:
# Save the arrays, fold assignments, paths, and manifests for Notebooks 03 and 04.
np.savez_compressed(
    OUT / "speaker_split_mfcc_10fold.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    train_paths=train_df["path"].to_numpy(),
    test_paths=test_df["path"].to_numpy(),
    train_speakers=train_df["speaker_id"].to_numpy(),
    test_speakers=test_df["speaker_id"].to_numpy(),
    fold_assignments=fold_assignments,
    mean=mean,
    std=std,
    classes=classes
)

joblib.dump(encoder, OUT / "label_encoder_10fold.joblib")
train_df.assign(cv_fold=fold_assignments).to_csv(OUT / "train_manifest_10fold.csv", index=False)
test_df.to_csv(OUT / "test_manifest_10fold.csv", index=False)
fold_summary.to_csv(OUT / "10fold_partition_summary.csv", index=False)

print("Saved 10-fold MFCC dataset and manifests in:", OUT.resolve())

Saved 10-fold MFCC dataset and manifests in: C:\Users\ADMIN\Desktop\Sukriti\VIT\SL\AnuAssistant\Anu_Revision_Notebooks\anu_revision_output


### Output files

- `speaker_split_mfcc_10fold.npz` — MFCC arrays, labels, speaker IDs, test partition, and fold assignments.
- `train_manifest_10fold.csv` — outer training data with fold number.
- `test_manifest_10fold.csv` — untouched final test data.
- `10fold_partition_summary.csv` — number of files and speakers in each fold.

Use these files in Notebooks 03 and 04. Do not regenerate the split between model comparisons.